In [7]:
import os
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [8]:
import pandas as pd

In [9]:
UEA_MTSC30 = ['EthanolConcentration',
              'FaceDetection',
              'Handwriting',
              'Heartbeat',
              'JapaneseVowels',
              'PEMS-SF',
              'SelfRegulationSCP1',
              'SelfRegulationSCP2',
              'SpokenArabicDigits',
              'UWaveGestureLibrary',
              'ArticularyWordRecognition',
              'AtrialFibrillation',
              'BasicMotions',
              'CharacterTrajectories',
              'Cricket',
              'DuckDuckGeese',
              'EigenWorms',
              'Epilepsy',
              'ERing',
              'FingerMovements',
              'HandMovementDirection',
              'InsectWingbeat',
              'Libras',
              'LSST',
              'MotorImagery',
              'NATOPS',
              'PenDigits',
              'PhonemeSpectra',
              'RacketSports',
              'StandWalkJump']

In [ ]:
ckpt_dir = '/data/user/MambaSL/checkpoints/'
best_ckpt_dir = '/data/user/MambaSL/checkpoints_final/'
best_script_dir = '/data/user/MambaSL/scripts_final/'

for model in [
    # 'DLinear', 'LightTS', 'MTSMixer', 'TimesNet', 
    # 'FEDformer', 'ETSformer', 'Crossformer', 'PatchTST', 'GPT4TS', 'iTransformer',
    # 'ModernTCN', 'TimeMixerPP', 'InterpretGN',
    # 'TSCMamba', 'Mamba'
    ]:
    print(f'==================={model}===================')
    os.makedirs(os.path.join(best_ckpt_dir, model), exist_ok=True)

    for data_name in UEA_MTSC30:

        if os.path.exists(f'../03-full_results/{model}/{model}_CLS_{data_name}.out'):
            with open(f'../03-full_results/{model}/{model}_CLS_{data_name}.out', 'r') as file:
                data = file.read().splitlines()
            with open(f'../03-full_results/{model}/{model}_CLS_{data_name}.sh', 'r') as file:
                scripts = file.read().split('\n\n')[:-1]
                scripts = list(filter(lambda x: x[0] != '#', scripts))
        else:
            # print('no file')
            continue


        print(data_name)
        result_lst = list()
        for i in range(len(data)):
            if data[i].startswith('>>>>>>>testing : '):
                ckpt = data[i].replace('>>>>>>>testing : ', '').replace('<', '').strip()
                data_meta = ckpt.split('_')

                if data[i+2].startswith('gating_value'):
                    acc = float(data[i+3].replace('accuracy:', ''))
                    model_params = data[i+4].replace('model parameter : ', '')
                    model_size = data[i+5].replace('model size : ', '').replace('MB', '')
                else:
                    acc = float(data[i+2].replace('accuracy:', ''))
                    model_params = data[i+3].replace('model parameter : ', '')
                    model_size = data[i+4].replace('model size : ', '').replace('MB', '')

                result_data = {
                    i: data_meta[i] for i in range(9, len(data_meta))
                }
                result_data.update({
                    'acc': float(acc),
                    'model_params': int(model_params),
                    'model_size (MB)': float(model_size),
                    'ckpt': ckpt
                })

                result_lst.append(result_data)

        result_df = pd.DataFrame(reversed(result_lst))
        scripts = list(reversed(scripts))
        
        print(result_df['acc'].max() * 100, len(result_df))
        result_acc_max = result_df[result_df['acc'] == result_df['acc'].max()].sort_values(['model_size (MB)', 'model_params'])
        # display(result_acc_max)



        # select best accuracy model (up to 20)
        result_acc_max = result_acc_max.iloc[:20]

        # move 20 lowest sized ckpts
        for ckpt in result_acc_max['ckpt']:
            if ckpt in os.listdir(ckpt_dir):
                os.makedirs(os.path.join(best_ckpt_dir, model), exist_ok=True)
                os.rename(os.path.join(ckpt_dir, ckpt), os.path.join(best_ckpt_dir, model, ckpt))
                print(f'> moved {ckpt}')
                

        assert len(result_df) == len(scripts), f'len(result_acc_max) != len(scripts) ({len(result_acc_max)} != {len(scripts)})'
        # make the scripts
        scripts_acc_max = f'''model_name="{model}"
dataset_name="{data_name}"
source_dir="/data/user/MambaSL"
gpu_id=0

data_dir="${{source_dir}}/dataset"
checkpoint_dir="${{source_dir}}/checkpoints_best/${{model_name}}"'''
        if len(result_acc_max) > 1:
            scripts_acc_max += '\n\n# below all have the same performance'

        for idx in result_acc_max.index:
            scripts_acc_max += '''\n\npython run.py \\
--use_gpu True \\
--gpu_type cuda \\
--gpu ${gpu_id} \\
--task_name classification \\
--data UEA \\
--root_path "${data_dir}/${dataset_name}" \\
--checkpoints "${checkpoint_dir}" \\
--model "${model_name}" \\
--model_id "CLS_${dataset_name}" \\
'''
            scripts_acc_max += '\n'.join(scripts[idx].replace("--is_training 1", "--is_training 0").split('\n')[15:])
        scripts_acc_max = scripts_acc_max.strip()
        out_dir = os.path.join(best_script_dir, model)
        script_fname = f'{out_dir}/{data_name}.sh'
        if not os.path.exists(script_fname):
            os.makedirs(out_dir, exist_ok=True)
            with open(script_fname, 'w') as file:
                file.write(scripts_acc_max)
            print(f'> saved {data_name}.sh')
            print()
    
        
        

In [ ]:
ckpt_dir = '/data/user/MambaSL/checkpoints/'
best_ckpt_dir = '/data/user/MambaSL/checkpoints_final/'
best_script_dir = '/data/user/MambaSL/scripts_final/'

model = 'MambaSL'
for subdir, exp in [
    ('_proposed', 'proposed'),
    # ('ablation1', 'ablation1'),
    # ('ablation2', 'ablation2'),
    # ('ablation3', 'ablation3'),
    # ('ablation4', 'ablation4'),
    # ('ablation5', 'ablation5'),
    # ('additional (inceptiontime-setting)', 'trainlossonly'),
    ]:
    print(f'==================={model}===================')
    os.makedirs(os.path.join(best_ckpt_dir, model, subdir), exist_ok=True)

    for data_name in UEA_MTSC30:

        if os.path.exists(f'../03-full_results/{model}/{subdir}/{model}_CLS_{data_name}_{exp}.out'):
            with open(f'../03-full_results/{model}/{subdir}/{model}_CLS_{data_name}_{exp}.out', 'r') as file:
                data = file.read().splitlines()
            with open(f'../03-full_results/{model}/{subdir}/{model}_CLS_{data_name}_{exp}.sh', 'r') as file:
                scripts = file.read().split('\n\n')[:-1]
                scripts = list(filter(lambda x: x[0] != '#', scripts))
        else:
            # print('no file')
            continue


        print(data_name)
        result_lst = list()
        for i in range(len(data)):
            if data[i].startswith('>>>>>>>testing : '):
                ckpt = data[i].replace('>>>>>>>testing : ', '').replace('<', '').strip()
                data_meta = ckpt.split('_')

                if data[i+2].startswith('gating_value'):
                    acc = float(data[i+3].replace('accuracy:', ''))
                    model_params = data[i+4].replace('model parameter : ', '')
                    model_size = data[i+5].replace('model size : ', '').replace('MB', '')
                else:
                    acc = float(data[i+2].replace('accuracy:', ''))
                    model_params = data[i+3].replace('model parameter : ', '')
                    model_size = data[i+4].replace('model size : ', '').replace('MB', '')

                result_data = {
                    i: data_meta[i] for i in range(9, len(data_meta))
                }
                result_data.update({
                    'acc': float(acc),
                    'model_params': int(model_params),
                    'model_size (MB)': float(model_size),
                    'ckpt': ckpt
                })

                result_lst.append(result_data)

        result_df = pd.DataFrame(reversed(result_lst))
        scripts = list(reversed(scripts))
        
        print(result_df['acc'].max() * 100, len(result_df))
        result_acc_max = result_df[result_df['acc'] == result_df['acc'].max()].sort_values(['model_size (MB)', 'model_params'])
        # display(result_acc_max)



        # select best accuracy model (up to 20)
        result_acc_max = result_acc_max.iloc[:20]

        # move 20 lowest sized ckpts
        for ckpt in result_acc_max['ckpt']:
            if ckpt in os.listdir(ckpt_dir):
                os.makedirs(os.path.join(best_ckpt_dir, model, subdir), exist_ok=True)
                os.rename(os.path.join(ckpt_dir, ckpt), os.path.join(best_ckpt_dir, model, subdir, ckpt))
                print(f'> moved {ckpt}')
                

        # assert len(result_df) == len(scripts), f'len(result_df) == len(scripts ({len(result_df)} != {len(scripts)})'
        if len(result_df) != len(scripts):
            print(f'{data_name} : len(result_df) != len(scripts) ({len(result_df)} != {len(scripts)})')

        # make the scripts
        scripts_acc_max = f'''model_name="MambaSingleLayer"
dataset_name="{data_name}"
source_dir="/data/user/MambaSL"
gpu_id=0

data_dir="${{source_dir}}/dataset"
checkpoint_dir="${{source_dir}}/checkpoints_best/${{model_name}}"'''
        if len(result_acc_max) > 1:
            scripts_acc_max += '\n\n# below all have the same performance'

        for idx in result_acc_max.index:
            scripts_acc_max += '''\n\npython run.py \\
  --use_gpu True \\
  --gpu_type cuda \\
  --gpu ${gpu_id} \\
  --task_name classification \\
  --data UEA \\
  --root_path "${data_dir}/${dataset_name}" \\
  --checkpoints "${checkpoint_dir}" \\
  --model "${model_name}" \\
  --model_id "CLS_${dataset_name}" \\
'''
            scripts_acc_max += '\n'.join(scripts[idx].replace("--is_training 1", "--is_training 0").split('\n')[15:])
        scripts_acc_max = scripts_acc_max.strip()
        out_dir = os.path.join(best_script_dir, model, subdir)
        script_fname = f'{out_dir}/{data_name}.sh'
        if not os.path.exists(script_fname):
            os.makedirs(out_dir, exist_ok=True)
            with open(script_fname, 'w') as file:
                file.write(scripts_acc_max)
            print(f'> saved {data_name}.sh')
            print()
    
        
        